# Test SmartNoiseModelLoader
This notebook validates the `SmartNoiseModelLoader` synthesizer, focusing on metadata preservation and reproducibility.

In [ ]:
%load_ext autoreload
%autoreload 2

import hydra
import pandas as pd
import numpy as np
from pathlib import Path
import torch

from src.loading.file_loader import FileLoader
from src.synthesizing.model_loader import SmartNoiseModelLoader
from src.entities.dataset import Dataset

# Setup
model_path = "models/adult100_mst.pkl"
if not Path(model_path).exists():
    print("Training model for test...")
    # This would normally be done via script, but we assume it exists from previous turn
    pass

# Load reference dataset
with hydra.initialize(version_base=None, config_path="../config"):
    cfg = hydra.compose(config_name="config", overrides=["loading=adult", "loading.size=100"])
    loader = hydra.utils.instantiate(cfg.loading)
    dataset = loader.load()
    dataset.mappings = {"test": "mapping"} # Mock mapping

print(f"Loaded dataset: {dataset.name}, size: {len(dataset)}")

## 1. Basic Generation

In [ ]:
loader_synth = SmartNoiseModelLoader(model_path=model_path, size=50, seed=42)
synthetic_ds = loader_synth.synthesize(dataset)

print(f"Synthetic dataset: {synthetic_ds.name}, size: {len(synthetic_ds)}")
assert len(synthetic_ds) == 50
assert synthetic_ds.mappings == dataset.mappings
display(synthetic_ds.data.head())

## 2. Reproducibility Test
Generate twice with the same seed and compare.

In [ ]:
synth1 = SmartNoiseModelLoader(model_path=model_path, size=50, seed=123).synthesize(dataset)
synth2 = SmartNoiseModelLoader(model_path=model_path, size=50, seed=123).synthesize(dataset)

pd.testing.assert_frame_equal(synth1.data, synth2.data)
print("Reproducibility with same seed: PASSED")

## 3. Different Seed Test
Generate with different seeds and ensure results differ.

In [ ]:
synth3 = SmartNoiseModelLoader(model_path=model_path, size=50, seed=456).synthesize(dataset)

try:
    pd.testing.assert_frame_equal(synth1.data, synth3.data)
    print("Different seeds test: FAILED (data is identical)")
except AssertionError:
    print("Different seeds test: PASSED (data is different)")